# TE02_026 - ROS Action Request Reception and Processing Validation

This notebook validates TE02_026: successful reception and processing of ROS action requests with a success rate greater than 90%.

The selected action server is `/action/effector_controller` using action type `telehandler_moveit/action/EffectorController`.

## Validation Method

The notebook acts as a ROS 2 action client. It sends a configurable number of goals to the effector controller action server and records whether the action communication path receives, accepts, and returns a response for each request.

A request is counted as a successful communication transaction when:

- the action server is available,
- the goal is accepted,
- the result response is received before timeout.

The KPI communication success rate is computed as:

`communication_success_rate = communication_successful_requests / total_requests * 100`

The KPI passes when `communication_success_rate > 90%`. The action result field `success` is recorded as diagnostic information only and is not used for the TE02_026 pass/fail decision.

## Workflow

1. Start the system containing the `/action/effector_controller` action server.
2. Make sure TF is available for the configured target and reference frames.
3. Adjust the goal configuration if needed.
4. Run the acquisition cell to send repeated action requests.
5. Run the summary cell to compute the KPI result.

Default goal configuration uses `auto_adjust=True`. If the current TF error is already below the controller threshold, each goal should complete quickly without causing significant motion.

In [1]:
from pathlib import Path
import time
import os
import numpy as np
import pandas as pd
import rclpy
from rclpy.action import ActionClient
from rclpy.node import Node

try:
    from telehandler_moveit.action import EffectorController
    ACTION_TYPE_SOURCE = 'telehandler_moveit.action.EffectorController'
except ImportError:
    from criarte_msgs.action import EffectorController
    ACTION_TYPE_SOURCE = 'criarte_msgs.action.EffectorController'

KPI_DIR = Path(str(os.environ.get('HOME')) + '/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_026')
KPI_DIR.mkdir(parents=True, exist_ok=True)

REQUESTS_CSV = KPI_DIR / 'te02_026_action_requests.csv'
FEEDBACK_CSV = KPI_DIR / 'te02_026_action_feedback.csv'
SUMMARY_CSV = KPI_DIR / 'te02_026_summary.csv'

ACTION_NAME = '/action/effector_controller'
TOTAL_REQUESTS = 30
COMMUNICATION_SUCCESS_THRESHOLD_PCT = 90.0
SERVER_WAIT_TIMEOUT_S = 10.0
RESULT_TIMEOUT_S = 30.0
INTER_REQUEST_DELAY_S = 1.0

# EffectorController goal fields. Adjust these if the active system uses different frames.
GOAL_FRAME_TARGET = 'arm_fork_link'
GOAL_FRAME_REFERENCE = 'basket_link'
GOAL_HEIGHT_DISTANCE = 0.0
GOAL_ANGLE_DEGREES = -10.0
GOAL_MAX_VELOCITY = 1.0
GOAL_AUTO_ADJUST = False

print(f'Output directory: {KPI_DIR}')
print(f'Action server: {ACTION_NAME}')
print(f'Action type source: {ACTION_TYPE_SOURCE}')
print(f'Total requests: {TOTAL_REQUESTS}')

Output directory: /home/lucaspinacosta/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_026
Action server: /action/effector_controller
Action type source: telehandler_moveit.action.EffectorController
Total requests: 30


In [2]:
class EffectorActionKPIClient(Node):
    def __init__(self):
        super().__init__('te02_026_effector_action_kpi_client')
        self.client = ActionClient(self, EffectorController, ACTION_NAME)
        self.feedback_rows = []

    def make_goal(self):
        goal = EffectorController.Goal()
        goal.frame_target = GOAL_FRAME_TARGET
        goal.frame_reference = GOAL_FRAME_REFERENCE
        goal.height_distance = float(GOAL_HEIGHT_DISTANCE)
        goal.angle_degrees = float(GOAL_ANGLE_DEGREES)
        goal.max_velocity = float(GOAL_MAX_VELOCITY)
        goal.auto_adjust = bool(GOAL_AUTO_ADJUST)
        return goal

    def feedback_cb(self, request_index):
        def _cb(feedback_msg):
            now_ns = self.get_clock().now().nanoseconds
            fb = feedback_msg.feedback
            self.feedback_rows.append({
                'request_index': request_index,
                'time_ns': now_ns,
                'time_s': now_ns / 1e9,
                'robot_state': getattr(fb, 'robot_state', ''),
                'current_frame': getattr(fb, 'current_frame', ''),
            })
            print(f"{fb}", flush=True, end='\n')
        return _cb

def spin_until_future(node, future, timeout_s):
    deadline = time.monotonic() + timeout_s
    while rclpy.ok() and not future.done() and time.monotonic() < deadline:
        rclpy.spin_once(node, timeout_sec=0.1)
    return future.done()

In [3]:
if not rclpy.ok():
    rclpy.init()

node = EffectorActionKPIClient()
request_rows = []

server_available = node.client.wait_for_server(timeout_sec=SERVER_WAIT_TIMEOUT_S)
print(f'Action server available: {server_available}')

try:
    for request_index in range(1, TOTAL_REQUESTS + 1):
        start_ns = node.get_clock().now().nanoseconds
        row = {
            'request_index': request_index,
            'action_name': ACTION_NAME,
            'action_type_source': ACTION_TYPE_SOURCE,
            'goal_frame_target': GOAL_FRAME_TARGET,
            'goal_frame_reference': GOAL_FRAME_REFERENCE,
            'goal_height_distance': GOAL_HEIGHT_DISTANCE,
            'goal_angle_degrees': GOAL_ANGLE_DEGREES,
            'goal_max_velocity': GOAL_MAX_VELOCITY,
            'goal_auto_adjust': GOAL_AUTO_ADJUST,
            'send_time_ns': start_ns,
            'server_available': server_available,
            'goal_accepted': False,
            'result_received': False,
            'communication_success': False,
            'result_success': False,
            'result_status': np.nan,
            'outcome': '',
            'latency_ms': np.nan,
            'status': 'FAIL',
            'reason': '',
        }

        if not server_available:
            row['reason'] = 'Action server was not available.'
            request_rows.append(row)
            continue

        goal = node.make_goal()
        send_future = node.client.send_goal_async(goal, feedback_callback=node.feedback_cb(request_index))
        if not spin_until_future(node, send_future, RESULT_TIMEOUT_S):
            row['reason'] = 'Timed out waiting for goal response.'
            request_rows.append(row)
            continue

        goal_handle = send_future.result()
        row['goal_accepted'] = bool(goal_handle.accepted)
        if not goal_handle.accepted:
            row['reason'] = 'Goal rejected by action server.'
            request_rows.append(row)
            continue

        result_future = goal_handle.get_result_async()
        if not spin_until_future(node, result_future, RESULT_TIMEOUT_S):
            row['reason'] = 'Timed out waiting for action result.'
            request_rows.append(row)
            continue

        result_response = result_future.result()
        finish_ns = node.get_clock().now().nanoseconds
        result = result_response.result
        row['result_received'] = True
        row['result_status'] = int(result_response.status)
        row['communication_success'] = True
        row['result_success'] = bool(result.success)
        row['outcome'] = str(result.outcome)
        row['latency_ms'] = (finish_ns - start_ns) / 1e6
        row['status'] = 'PASS'
        row['reason'] = f'Communication completed; action_result_success={row["result_success"]}; outcome={row["outcome"]}'
        request_rows.append(row)

        print(f'Request {request_index}/{TOTAL_REQUESTS}: {row["status"]} - {row["reason"]}')
        if INTER_REQUEST_DELAY_S > 0:
            time.sleep(INTER_REQUEST_DELAY_S)
finally:
    requests_df = pd.DataFrame(request_rows)
    feedback_df = pd.DataFrame(node.feedback_rows)
    node.destroy_node()

requests_df.to_csv(REQUESTS_CSV, index=False)
feedback_df.to_csv(FEEDBACK_CSV, index=False)
print(f'Wrote requests: {REQUESTS_CSV}')
print(f'Wrote feedback: {FEEDBACK_CSV}')
display(requests_df.head())

Action server available: True
telehandler_moveit.action.EffectorController_Feedback(robot_state='planning', current_frame='basket_link_to_armFork_joint')
telehandler_moveit.action.EffectorController_Feedback(robot_state='executing', current_frame='basket_link_to_armFork_joint')
telehandler_moveit.action.EffectorController_Feedback(robot_state='pt 1/4 err=0.0000 tol=0.0400', current_frame='basket_link_to_armFork_joint')
telehandler_moveit.action.EffectorController_Feedback(robot_state='pt 2/4 err=0.1108 tol=0.0400', current_frame='basket_link_to_armFork_joint')
telehandler_moveit.action.EffectorController_Feedback(robot_state='pt 2/4 err=0.1108 tol=0.0400', current_frame='basket_link_to_armFork_joint')
telehandler_moveit.action.EffectorController_Feedback(robot_state='pt 2/4 err=0.1108 tol=0.0400', current_frame='basket_link_to_armFork_joint')
telehandler_moveit.action.EffectorController_Feedback(robot_state='pt 2/4 err=0.1108 tol=0.0400', current_frame='basket_link_to_armFork_joint')
t

,request_index,action_name,action_type_source,goal_frame_target,goal_frame_reference,goal_height_distance,goal_angle_degrees,goal_max_velocity,goal_auto_adjust,send_time_ns,server_available,goal_accepted,result_received,communication_success,result_success,result_status,outcome,latency_ms,status,reason
0,1,/action/effector_controller,telehandler_moveit.action.EffectorController,arm_fork_link,basket_link,0.0,-10.0,1.0,False,1783517531627206351,True,True,True,True,True,4,Done.,32867.007697,PASS,Communication completed; action_result_success...
1,2,/action/effector_controller,telehandler_moveit.action.EffectorController,arm_fork_link,basket_link,0.0,-10.0,1.0,False,1783517565494340083,True,True,True,True,True,4,Done.,160.495578,PASS,Communication completed; action_result_success...
2,3,/action/effector_controller,telehandler_moveit.action.EffectorController,arm_fork_link,basket_link,0.0,-10.0,1.0,False,1783517566655119461,True,True,True,True,True,4,Done.,92.167073,PASS,Communication completed; action_result_success...
3,4,/action/effector_controller,telehandler_moveit.action.EffectorController,arm_fork_link,basket_link,0.0,-10.0,1.0,False,1783517567747520027,True,True,True,True,True,4,Done.,74.510802,PASS,Communication completed; action_result_success...
4,5,/action/effector_controller,telehandler_moveit.action.EffectorController,arm_fork_link,basket_link,0.0,-10.0,1.0,False,1783517568822189489,True,True,True,True,True,4,Done.,136.445292,PASS,Communication completed; action_result_success...


In [4]:
total_requests = int(len(requests_df))
server_available_count = int(requests_df['server_available'].sum()) if total_requests else 0
accepted_count = int(requests_df['goal_accepted'].sum()) if total_requests else 0
result_received_count = int(requests_df['result_received'].sum()) if total_requests else 0
communication_success_count = int(requests_df['communication_success'].sum()) if total_requests else 0
action_result_success_count = int(requests_df['result_success'].sum()) if total_requests else 0
communication_success_rate_pct = 100.0 * communication_success_count / total_requests if total_requests else 0.0
action_result_success_rate_pct = 100.0 * action_result_success_count / total_requests if total_requests else 0.0
accepted_rate_pct = 100.0 * accepted_count / total_requests if total_requests else 0.0
result_received_rate_pct = 100.0 * result_received_count / total_requests if total_requests else 0.0
latency_p95_ms = float(requests_df['latency_ms'].dropna().quantile(0.95)) if requests_df['latency_ms'].notna().any() else np.nan
overall_status = 'PASS' if communication_success_rate_pct > COMMUNICATION_SUCCESS_THRESHOLD_PCT else 'FAIL'

summary_rows = [
    {'metric': 'total_requests', 'value': total_requests, 'status': 'INFO', 'reason': ''},
    {'metric': 'server_available_requests', 'value': server_available_count, 'status': 'PASS' if server_available_count == total_requests else 'FAIL', 'reason': ''},
    {'metric': 'accepted_requests', 'value': accepted_count, 'status': 'PASS' if accepted_count == total_requests else 'FAIL', 'reason': f'accepted_rate_pct={accepted_rate_pct:.2f}'},
    {'metric': 'result_received_requests', 'value': result_received_count, 'status': 'PASS' if result_received_count == total_requests else 'FAIL', 'reason': f'result_received_rate_pct={result_received_rate_pct:.2f}'},
    {'metric': 'communication_successful_requests', 'value': communication_success_count, 'status': overall_status, 'reason': ''},
    {'metric': 'communication_success_rate_pct', 'value': communication_success_rate_pct, 'status': overall_status, 'reason': f'threshold>{COMMUNICATION_SUCCESS_THRESHOLD_PCT}%'},
    {'metric': 'action_result_successful_requests_diagnostic', 'value': action_result_success_count, 'status': 'INFO', 'reason': f'action_result_success_rate_pct={action_result_success_rate_pct:.2f}; not used for KPI pass/fail'},
    {'metric': 'latency_p95_ms', 'value': latency_p95_ms, 'status': 'INFO', 'reason': ''},
    {'metric': 'TE02_026_overall', 'value': overall_status, 'status': overall_status, 'reason': f'{communication_success_count}/{total_requests} action communication transactions completed'},
]

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_CSV, index=False)
display(summary_df)
print(f'Wrote summary: {SUMMARY_CSV}')
print(f'Overall TE02_026 status: {overall_status}')

,metric,value,status,reason
0,total_requests,30,INFO,
1,server_available_requests,30,PASS,
2,accepted_requests,30,PASS,accepted_rate_pct=100.00
3,result_received_requests,30,PASS,result_received_rate_pct=100.00
4,communication_successful_requests,30,PASS,
5,communication_success_rate_pct,100.0,PASS,threshold>90.0%
6,action_result_successful_requests_diagnostic,30,INFO,action_result_success_rate_pct=100.00; not use...
7,latency_p95_ms,901.781771,INFO,
8,TE02_026_overall,PASS,PASS,30/30 action communication transactions completed


Wrote summary: /home/lucaspinacosta/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_026/te02_026_summary.csv
Overall TE02_026 status: PASS


## Interpretation

Use `te02_026_action_requests.csv` as detailed evidence for each action request.

Use `te02_026_action_feedback.csv` as supporting evidence that the action server published feedback while processing requests.

Use `te02_026_summary.csv` as the KPI result table. TE02_026 passes when the action communication success rate is greater than 90%. The action result `success` field is kept as diagnostic information only.